# Notebook 21 VinDr SmallNodule

**VinDr-CXR small-nodule confirmatory analysis.**

VinDr-CXR small-nodule replication of NIH exp19.

Tests the small-feature hypothesis on a small-lesion-rich, bbox-annotated
external cohort. VinDr-CXR test set has 177 Nodule/Mass-positive images
out of 3,000 (5.9% prevalence) with per-image bounding-box annotations,
giving ~89/88 small-vs-large stratum sizes vs n=19/19 on the NIH BBox
subset — substantially better statistical power for the size-stratified
test of the small-feature claim.

Protocol matches NIH exp19:
  - fit Nodule/Mass linear probe on VinDr train clean embeddings
  - predict on VinDr test clean and VinDr test perturbed
  - per-image score-shift: Spearman rho between (clean - perturbed) score
    drop and log(lesion bbox area) on test positives
  - median-split positives by lesion area; compute stratified DeltaAUC
  - perturbations: dir_blur_cranio_p64 and reticular_fine_p32
    (the two manuscript exp19 rows we need to replicate)
  - models: raddino, dinov3 (matching the manuscript exp19 table)

Outputs (v4_work/v4_exp_vindr_smallnodule/):
  exp_vindr_smallnodule_strata.parquet  -- exp19 strata schema
  exp_vindr_smallnodule_corr.parquet    -- exp19 corr   schema
  exp_vindr_smallnodule_summary.parquet -- one row per (model, perturbation)


In [ ]:
# === Papermill parameters (override via `papermill -p NAME VALUE`) ===
DATASET = "nih"            # one of: nih, mimic, emory, vindr (where applicable)
SEED = 42
OUTPUTS_DIR = "/home/saptpurk/embeddings-noise-eliminators/outputs"
REPO_ROOT_OVERRIDE = "/home/saptpurk/embeddings-noise-eliminators"
HF_TOKEN_OVERRIDE = None     # set non-None when running gated models locally


In [ ]:
# Apply papermill parameters to environment
import os
os.environ.setdefault("DATASET", DATASET)
os.environ.setdefault("OUTPUTS_DIR", OUTPUTS_DIR)
os.environ.setdefault("REPO_ROOT", REPO_ROOT_OVERRIDE)
if HF_TOKEN_OVERRIDE:
    os.environ["HF_TOKEN"] = HF_TOKEN_OVERRIDE


In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import torch
from PIL import Image
from pydicom.pixel_data_handlers.util import apply_voi_lut
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

REPO_ROOT = Path(os.environ.get(
    "REPO_ROOT", "/home/saptpurk/embeddings-noise-eliminators"))
sys.path.insert(0, str(REPO_ROOT))

from common import (  # noqa: E402
    PARAMS,
    DirectionalMotionBlurInjector,
    EmbeddingExtractor,
    ReticularPatternInjector,
)

VINDR_ROOT = Path("/data0/vindr-cxr")
WORK = Path(os.environ.get(
    "OUTPUTS_DIR", "/home/saptpurk/embeddings-noise-eliminators/outputs"))
OUT = WORK / "v4_exp_vindr_smallnodule"
CACHE = OUT / "cache"
OUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

SEED = PARAMS.random_seed
TARGET_SIZE = (1024, 1024)
EXPECTED_TRAIN = 15000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PERTURBATIONS = [
    ("dir_blur_cranio_p64",
     DirectionalMotionBlurInjector(
         seed=SEED, angle_deg=90.0,
         kernel_length=PARAMS.directional_kernel_length),
     64),
    ("reticular_fine_p32",
     ReticularPatternInjector(seed=SEED, period_px=3, amplitude=0.08),
     32),
]
MODELS = ("raddino", "dinov3")


# ---------------------------------------------------------------------------
# 1. Wait for VinDr sync to finish
# ---------------------------------------------------------------------------

def wait_for_sync(min_train_dicoms: int = 14500, poll_sec: int = 30) -> None:
    train_dir = VINDR_ROOT / "train"
    print(f"Waiting for VinDr train/ to reach >= {min_train_dicoms} DICOMs ...")
    while True:
        n = sum(1 for _ in train_dir.glob("*.dicom")) if train_dir.exists() else 0
        sync_alive = int(os.popen(
            "pgrep -f 'aws s3 sync.*vindr' | wc -l").read().strip()) > 0
        print(f"  train DICOMs: {n}    sync_running: {sync_alive}")
        if n >= min_train_dicoms and not sync_alive:
            print("  sync complete.")
            return
        if n >= EXPECTED_TRAIN:
            print("  train at full count, proceeding even if sync flag stale.")
            return
        time.sleep(poll_sec)


# ---------------------------------------------------------------------------
# 2. DICOM -> uint8 1024x1024 grayscale loader
# ---------------------------------------------------------------------------

def load_dicom_1024(path: Path, target=TARGET_SIZE) -> np.ndarray:
    """Read a VinDr DICOM, apply VOI LUT if present, min-max normalize,
    return uint8 1024x1024 grayscale (H, W, 1)."""
    ds = pydicom.dcmread(str(path))
    arr = ds.pixel_array
    try:
        arr = apply_voi_lut(arr, ds)
    except Exception:
        pass
    arr = arr.astype(np.float32)
    if ds.get("PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
        arr = arr.max() - arr
    lo, hi = float(arr.min()), float(arr.max())
    if hi > lo:
        arr = (arr - lo) / (hi - lo)
    arr = (arr * 255.0).clip(0, 255).astype(np.uint8)
    img = Image.fromarray(arr, mode="L").resize(target, Image.LANCZOS)
    return np.array(img)[:, :, None]   # (H, W, 1)


# ---------------------------------------------------------------------------
# 3. Label tables
# ---------------------------------------------------------------------------

def load_label_tables():
    train_lbl = pd.read_csv(VINDR_ROOT / "annotations" / "image_labels_train.csv")
    test_lbl  = pd.read_csv(VINDR_ROOT / "annotations" / "image_labels_test.csv")
    test_bbox = pd.read_csv(VINDR_ROOT / "annotations" / "annotations_test.csv")

    # Train: aggregate radiologist votes -> any-positive on Nodule/Mass.
    train_y = (train_lbl.groupby("image_id")["Nodule/Mass"]
                        .max().reset_index()
                        .rename(columns={"Nodule/Mass": "y"}))
    # Test: already single-row consensus.
    test_y = test_lbl[["image_id", "Nodule/Mass"]].rename(
        columns={"Nodule/Mass": "y"})

    # Per-image bbox area for Nodule/Mass on test set.
    nm = test_bbox[test_bbox["class_name"].isin(["Nodule/Mass", "Nodule", "Mass"])].copy()
    nm["area"] = ((nm["x_max"] - nm["x_min"]).clip(lower=0)
                  * (nm["y_max"] - nm["y_min"]).clip(lower=0))
    bbox_area = (nm.groupby("image_id")["area"]
                   .sum().reset_index()
                   .rename(columns={"area": "lesion_area"}))

    train_files = {p.stem for p in (VINDR_ROOT / "train").glob("*.dicom")}
    test_files  = {p.stem for p in (VINDR_ROOT / "test").glob("*.dicom")}
    train_y = train_y[train_y["image_id"].isin(train_files)].reset_index(drop=True)
    test_y  = test_y[test_y["image_id"].isin(test_files)].reset_index(drop=True)
    test_y  = test_y.merge(bbox_area, on="image_id", how="left")
    return train_y, test_y


# ---------------------------------------------------------------------------
# 4. Embedding cache (CLS only)
# ---------------------------------------------------------------------------

def cache_path(model: str, split: str, pert: str) -> Path:
    return CACHE / f"emb_{model}_{split}_{pert}.npz"


def _make_extractor(model_name: str, hf_token):
    return EmbeddingExtractor(model_name=model_name, hf_token=hf_token,
                              device=str(DEVICE))


def extract_clean(extractor, model_name: str, df: pd.DataFrame, split: str,
                  batch_size: int = 16) -> np.ndarray:
    cp = cache_path(model_name, split, "clean")
    if cp.exists():
        return np.load(cp)["X"]
    print(f"\n[{model_name} {split}] extracting clean embeddings, n={len(df)}")
    feats = []
    src_dir = VINDR_ROOT / split
    batch_imgs = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"{model_name} {split} clean"):
        img = load_dicom_1024(src_dir / f"{row['image_id']}.dicom")
        img3 = np.repeat(img, 3, axis=2)   # HWC uint8 numpy, 3 channels
        batch_imgs.append(img3)
        if len(batch_imgs) >= batch_size:
            feats.append(extractor.extract_cls(batch_imgs))
            batch_imgs = []
    if batch_imgs:
        feats.append(extractor.extract_cls(batch_imgs))
    X = np.vstack(feats)
    np.savez_compressed(cp, X=X)
    return X


def extract_perturbed(extractor, model_name: str, df: pd.DataFrame, split: str,
                      pert_name: str, injector, patch_size: int,
                      batch_size: int = 16) -> np.ndarray:
    cp = cache_path(model_name, split, pert_name)
    if cp.exists():
        return np.load(cp)["X"]
    print(f"\n[{model_name} {split}] extracting perturbed ({pert_name}), n={len(df)}")
    feats = []
    src_dir = VINDR_ROOT / split
    batch_imgs = []
    for _, row in tqdm(df.iterrows(), total=len(df),
                       desc=f"{model_name} {split} {pert_name}"):
        clean = load_dicom_1024(src_dir / f"{row['image_id']}.dicom")
        clean2d = clean[:, :, 0]
        noisy2d, _ = injector(clean2d, patch_size=patch_size, num_patches=1,
                              image_path=str(row["image_id"]))
        img3 = np.repeat(noisy2d[:, :, None], 3, axis=2)
        batch_imgs.append(img3)
        if len(batch_imgs) >= batch_size:
            feats.append(extractor.extract_cls(batch_imgs))
            batch_imgs = []
    if batch_imgs:
        feats.append(extractor.extract_cls(batch_imgs))
    X = np.vstack(feats)
    np.savez_compressed(cp, X=X)
    return X


# ---------------------------------------------------------------------------
# 5. Probe + size-stratified analysis
# ---------------------------------------------------------------------------

def fit_probe(X_train: np.ndarray, y_train: np.ndarray):
    scaler = StandardScaler().fit(X_train)
    Xtr = scaler.transform(X_train)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    best_C, best_cv = None, -np.inf
    for C in PARAMS.lr_C_grid:
        cv = []
        for tr_idx, va_idx in skf.split(Xtr, y_train):
            clf = LogisticRegression(C=C, max_iter=PARAMS.lr_max_iter,
                                     class_weight="balanced",
                                     solver="lbfgs", random_state=SEED)
            clf.fit(Xtr[tr_idx], y_train[tr_idx])
            cv.append(roc_auc_score(
                y_train[va_idx],
                clf.predict_proba(Xtr[va_idx])[:, 1]))
        m = float(np.mean(cv))
        if m > best_cv:
            best_cv, best_C = m, C
    clf = LogisticRegression(C=best_C, max_iter=PARAMS.lr_max_iter,
                             class_weight="balanced",
                             solver="lbfgs", random_state=SEED)
    clf.fit(Xtr, y_train)
    return scaler, clf, float(best_C), float(best_cv)


def stratified_analysis(model_name: str, pert_name: str,
                        scaler, clf,
                        X_test_clean: np.ndarray, X_test_pert: np.ndarray,
                        test_y: pd.DataFrame):
    Xc = scaler.transform(X_test_clean)
    Xp = scaler.transform(X_test_pert)
    p_clean = clf.predict_proba(Xc)[:, 1]
    p_pert  = clf.predict_proba(Xp)[:, 1]

    df = test_y.copy().reset_index(drop=True)
    df["p_clean"]    = p_clean
    df["p_perturbed"] = p_pert
    df["score_drop"] = df["p_clean"] - df["p_perturbed"]

    pos = df[(df["y"] == 1) & df["lesion_area"].notna()
             & (df["lesion_area"] > 0)].copy()
    pos["log_area"] = np.log(pos["lesion_area"])

    # Spearman rho on positives
    if len(pos) >= 5:
        rho, p_value = stats.spearmanr(pos["score_drop"], pos["log_area"])
    else:
        rho, p_value = float("nan"), float("nan")
    corr_row = dict(model=model_name, perturbation=pert_name,
                    n=int(len(pos)), spearman_rho=float(rho),
                    p_value=float(p_value))

    # Stratified DeltaAUC on median split of positives
    median_area = float(pos["log_area"].median()) if len(pos) else float("nan")
    pos["stratum"] = np.where(pos["log_area"] <= median_area, "small", "large")
    full_clean_auc = float(roc_auc_score(df["y"], p_clean))
    full_pert_auc  = float(roc_auc_score(df["y"], p_pert))

    strata_rows = []
    neg = df[df["y"] == 0]
    for stratum, sub in pos.groupby("stratum"):
        y_sub = np.concatenate([np.ones(len(sub)), np.zeros(len(neg))])
        c = np.concatenate([sub["p_clean"].values, neg["p_clean"].values])
        p = np.concatenate([sub["p_perturbed"].values, neg["p_perturbed"].values])
        if len(np.unique(y_sub)) < 2:
            continue
        a_c = float(roc_auc_score(y_sub, c))
        a_p = float(roc_auc_score(y_sub, p))
        strata_rows.append(dict(
            dataset="vindr", model=model_name, perturbation=pert_name,
            stratum=stratum, n_pos=int(len(sub)),
            auc_clean=a_c, auc_perturbed=a_p, delta_auc=a_c - a_p,
            auc_clean_fulltest=full_clean_auc,
            auc_perturbed_fulltest=full_pert_auc,
            delta_auc_fulltest=full_clean_auc - full_pert_auc,
        ))

    # Overall summary row
    strata_df = pd.DataFrame(strata_rows)
    if len(strata_df) == 2:
        small = strata_df[strata_df["stratum"] == "small"].iloc[0]
        large = strata_df[strata_df["stratum"] == "large"].iloc[0]
        gap = float(small["delta_auc"] - large["delta_auc"])
    else:
        gap = float("nan")
    summary_row = dict(
        dataset="vindr", model=model_name, perturbation=pert_name,
        n_pos=int(len(pos)),
        spearman_rho=float(rho), spearman_p=float(p_value),
        auc_clean_fulltest=full_clean_auc,
        auc_perturbed_fulltest=full_pert_auc,
        delta_auc_fulltest=full_clean_auc - full_pert_auc,
        gap_small_minus_large=gap,
    )
    return strata_rows, corr_row, summary_row


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    wait_for_sync()
    train_y, test_y = load_label_tables()
    print(f"\nVinDr labels loaded: train={len(train_y)} (pos={int(train_y['y'].sum())}), "
          f"test={len(test_y)} (pos={int(test_y['y'].sum())}, "
          f"with_bbox={int(test_y['lesion_area'].notna().sum())})")
    hf_token = os.environ.get("HF_TOKEN")

    all_strata, all_corr, all_summary = [], [], []

    for model_name in MODELS:
        extractor = _make_extractor(model_name, hf_token)
        # Clean embeddings (one-time per model, cached)
        Xtr_clean = extract_clean(extractor, model_name, train_y, "train")
        Xte_clean = extract_clean(extractor, model_name, test_y,  "test")
        scaler, clf, best_C, cv_auc = fit_probe(Xtr_clean, train_y["y"].values)
        print(f"  [{model_name}] probe: best_C={best_C} cv_auc={cv_auc:.4f}")

        for pert_name, injector, patch_size in PERTURBATIONS:
            Xte_pert = extract_perturbed(extractor, model_name, test_y, "test",
                                         pert_name, injector, patch_size)
            strata_rows, corr_row, summary_row = stratified_analysis(
                model_name, pert_name, scaler, clf,
                Xte_clean, Xte_pert, test_y)
            print(f"  [{model_name} {pert_name}] n_pos={summary_row['n_pos']} "
                  f"rho={summary_row['spearman_rho']:+.3f} (p={summary_row['spearman_p']:.3g}) "
                  f"gap_S-L={summary_row['gap_small_minus_large']:+.4f}")
            all_strata.extend(strata_rows)
            all_corr.append(corr_row)
            all_summary.append(summary_row)

            # Per-perturbation checkpoint
            pd.DataFrame(all_strata).to_parquet(
                OUT / "exp_vindr_smallnodule_strata.parquet", index=False)
            pd.DataFrame(all_corr).to_parquet(
                OUT / "exp_vindr_smallnodule_corr.parquet", index=False)
            pd.DataFrame(all_summary).to_parquet(
                OUT / "exp_vindr_smallnodule_summary.parquet", index=False)

        # Free GPU memory before loading the next model.
        del extractor
        torch.cuda.empty_cache()

    print("\n=== DONE ===")
    print(pd.DataFrame(all_summary).to_string(index=False))


# === Main entry-point (papermill will execute) ===
    main()
